# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content-day** (a single article's performance on a single day for one client).

| Property | Value |
|---|---|
| **Table** | `fact_content_daily_performance` (Hugging Face, ~79M rows) |
| **Grain columns** | `report_date` × `client_hash_id` × `content_hash_id` |
| **Panel span** | 2025-01-27 → 2026-06-30 (~17 months) |
| **Analysis month** | `month=2026-03` (mid-panel, not the final month) — avoids leaking the natural outcome window |
| **Feature window** | 30 days before the label window (Jan 30 – Feb 28) — knowable at prediction time |
| **Label window** | The most recent 30 days (March 2026) — e.g. "did impressions drop ≥20%?" |
| **Dataset** | starter CSV (`content_refresh_anonymized.csv`) for prototyping; warehouse for final verification |

**Why this grain:** An editor decides which *page* to review *next*. The daily panel lets me build time-series features (trends, momentum) that a single snapshot can't capture. Each content-day row carries search impressions, clicks, position, and engagement metrics that I can aggregate per content item for the scoring model.

**Why a mid-panel month:** The card warns that `_sample` is the final month (June 2026) — developing label logic there would train inside the test window. Month `2026-03` gives enough history before and after for a proper feature/label split.

In [1]:
# Section 1 — Connect to the warehouse and verify the grain + counts

# 1a. Install dependencies (run once)
# %pip -q install duckdb huggingface_hub scikit-learn pandas

import os, getpass, duckdb, pandas as pd
import numpy as np

# 1b. Authenticate — set HF_TOKEN as an env var or use the safe prompt
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

# 1c. Connect DuckDB to the hosted release
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_DAILY = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
FACT_QUERY = f"read_parquet('{REL}/fact_content_query_90d.parquet')"

# 1d. Row count and date span for the full table
full_info = con.sql(f"""
    SELECT 
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {FACT_DAILY}
""").df()

print('=== Full table ===')
print(f"Total rows: {full_info['total_rows'][0]:,}")
print(f"Date range: {full_info['first_date'][0]} to {full_info['last_date'][0]}")

# 1e. Row count for the analysis month (mid-panel: 2026-03)
mar_info = con.sql(f"""
    SELECT 
        COUNT(*) AS march_rows,
        COUNT(DISTINCT content_hash_id) AS distinct_content,
        COUNT(DISTINCT client_hash_id) AS distinct_clients
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()

print()
print('=== March 2026 (analysis month) ===')
print(f"Rows:           {mar_info['march_rows'][0]:,}")
print(f"Content items:  {mar_info['distinct_content'][0]:,}")
print(f"Clients:        {mar_info['distinct_clients'][0]}")

# 1f. Grain check — should return 0 rows
grain_violations = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS cnt
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print()
print('=== Grain check (should be 0 rows) ===')
print(f"Grain violations (report_date × client × content > 1 row): {len(grain_violations)}")
if len(grain_violations) > 0:
    print(grain_violations)

=== Full table ===
Total rows: 78,835,655
Date range: 2025-01-27 00:00:00 to 2026-06-30 00:00:00

=== March 2026 (analysis month) ===
Rows:           9,841,378
Content items:  331,437
Clients:        55

=== Grain check (should be 0 rows) ===
Grain violations (report_date × client × content > 1 row): 0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features (knowable before the decision moment)

These are columns from the daily fact that I will aggregate per content item into a scoring feature set. All are "knowable at the decision moment" because they come from a trailing window that ends *before* the label window.

| Field | Source | Knowable because |
|---|---|---|
| `gsc_impressions` (lagged, prev-30d sum) | `fact_content_daily_performance` | Aggregated from Jan 30 – Feb 28 — closed before March (label window) starts |
| `gsc_clicks` (lagged, prev-30d sum) | `fact_content_daily_performance` | Same closed window as impressions |
| `gsc_avg_position` (lagged, prev-30d avg) | `fact_content_daily_performance` | Same lagged window — position data available as soon as the day ends |
| `days_with_impressions` (prev-30d count) | `fact_content_daily_performance` | Count of days with ≥1 impression in the feature window |
| `content_age_days` (computed) | `dim_content.content_created_date` | Computed from publish date — known and static |
| `word_count` | `dim_content` | Known at publish — static over the article's life |

### Label / proxy

| Field | Definition | Notes |
|---|---|---|
| `is_declining` | 1 if `SUM(impressions_last_30d) < 0.8 × SUM(impressions_prev_30d)`, else 0 | This is a **proxy** label — it measures "currently declining relative to last month", not "will decline next month". It's the warehouse-scale equivalent of the starter CSV's `is_declining_label`. |

### Context (grouping / joining / splitting — never model features)

| Field | Use |
|---|---|
| `content_hash_id` | Join key between fact and dim tables. Use for per-content grouping |
| `client_hash_id` | Split key for client-holdout validation. Never a feature |
| `report_date` | Partition filter and time-window definition. Never a feature |

### Excluded (with reasons)

| Field | Why excluded |
|---|---|
| `gsc_impressions` from the *same* 30-day window as the label | **Leakage** — if the label is "decline in last 30d", then impressions from those same 30 days are the label, not a feature. |
| `content_hash_id` as a raw feature | **Pseudonym** — it's a random hash, not a meaningful number. The model would memorize IDs, not generalize. |
| `client_hash_id` as a raw feature | **Same reason** — memorizes clients instead of learning patterns that cross clients. |
| Any URL or client name | **Private data** — never appears in the pseudonymized release. |

### Output

The analysis hands the editor a **ranked queue of content items** sorted by declining probability, with Precision@50 as the success metric — same as ML-03, now built from warehouse-scale daily features instead of the starter CSV snapshot.

In [2]:
# Section 2 — Verify the fields exist in their respective tables

# DuckDB needs DESCRIBE SELECT * FROM (not DESCRIBE read_parquet directly)
print('=== Columns in fact_content_daily_performance (first 25) ===')
print(con.sql(f"DESCRIBE SELECT * FROM {FACT_DAILY}").df().head(25).to_string(index=False))

print()
print('=== Columns in dim_content ===')
print(con.sql(f"DESCRIBE SELECT * FROM {DIM_CONTENT}").df().to_string(index=False))

print()
print('=== Columns in dim_clients ===')
print(con.sql(f"DESCRIBE SELECT * FROM {DIM_CLIENTS}").df().to_string(index=False))

=== Columns in fact_content_daily_performance (first 25) ===
             column_name column_type null  key default extra
             report_date        DATE  YES None    None  None
          client_hash_id     VARCHAR  YES None    None  None
         content_hash_id     VARCHAR  YES None    None  None
          client_has_gsc     BOOLEAN  YES None    None  None
          client_has_ga4     BOOLEAN  YES None    None  None
      gsc_data_available     BOOLEAN  YES None    None  None
      ga4_data_available     BOOLEAN  YES None    None  None
         gsc_impressions      BIGINT  YES None    None  None
              gsc_clicks      BIGINT  YES None    None  None
        gsc_sum_position      BIGINT  YES None    None  None
        gsc_avg_position      DOUBLE  YES None    None  None
           ga4_pageviews      BIGINT  YES None    None  None
            ga4_sessions      BIGINT  YES None    None  None
               ga4_users      BIGINT  YES None    None  None
    ga4_engaged_sessions

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification A — Grain (already done in Section 1 — 0 violations ✓)

### Verification B — Availability: GA4 data available vs not available

The skill says: *"Rows before a client's ga4_data_start have GA4 columns zero-filled with ga4_data_available = FALSE — filter on the flag."* I will check how many rows have GA4 available vs not in my analysis month.

### Verification C — Five features with "knowable at decision moment" justification

I will build a small feature set from the March 2026 slice to prove I can pull the data.

### Verification D — The leakage trap

Per the card: *"add ONE label-derived column on purpose, watch your quick score jump toward perfect, then delete it and keep the honest number."*

I will:
1. Build a baseline model on clean features (from the prev-30d window)
2. Add a leaked column (impressions from the same window that defines the label)
3. Show the score jump
4. Delete it and keep the honest baseline

In [3]:
# Verification B — GA4 availability in March 2026

avail = con.sql(f"""
    SELECT 
        ga4_data_available,
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS clients,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
    FROM {FACT_DAILY}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    GROUP BY ga4_data_available
    ORDER BY ga4_data_available
""").df()

print('=== GA4 data availability in March 2026 ===')
print(avail.to_string(index=False))
print()
print('Note: ga4_data_available can be TRUE, FALSE, or NULL.')
print('NULL rows need IS NOT TRUE filtering, not just = FALSE.')

=== GA4 data availability in March 2026 ===
 ga4_data_available  row_count  clients  pct
              False    6408671       43 65.1
               True     413966       41  4.2
               <NA>    3018741       22 30.7

Note: ga4_data_available can be TRUE, FALSE, or NULL.
NULL rows need IS NOT TRUE filtering, not just = FALSE.


In [4]:
# Verification C — Build five features with "knowable because" justification
# Feature window: Jan 30 – Feb 28 (30 days before March)
# Label window: March 1 – March 31 (the analysis month)

features_df = con.sql(f"""
    WITH per_content AS (
        SELECT 
            f.content_hash_id,
            f.client_hash_id,
            -- Feature 1: Impressions from prev-30d (Jan 30 – Feb 28)
            SUM(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' 
                     THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            -- Feature 2: Clicks from prev-30d
            SUM(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' 
                     THEN f.gsc_clicks ELSE 0 END) AS clk_prev30,
            -- Feature 3: Average position from prev-30d
            AVG(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' 
                     THEN f.gsc_avg_position END) AS pos_prev30,
            -- Feature 4: Days with impressions in prev-30d (consistency signal)
            COUNT(DISTINCT CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' 
                                AND f.gsc_impressions > 0 THEN f.report_date END) AS days_with_imp_prev30,
            -- LABEL: Impressions in March (last 30d) — for computing is_declining
            SUM(CASE WHEN f.report_date >= DATE '2026-03-01' AND f.report_date <= DATE '2026-03-31' 
                     THEN f.gsc_impressions ELSE 0 END) AS imp_last30
        FROM {FACT_DAILY} f
        WHERE f.report_date >= DATE '2026-01-30' AND f.report_date <= DATE '2026-03-31'
          AND f.ga4_data_available IS TRUE
        GROUP BY f.content_hash_id, f.client_hash_id
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM per_content
""").df()

# Feature 5: content_age_days — computed from content_created_date in dim_content
# The warehouse dim_content has content_created_date, not content_age_days
content_meta = con.sql(f"""
    SELECT 
        content_hash_id, 
        DATEDIFF('day', content_created_date, DATE '2026-03-01') AS content_age_days,
        word_count
    FROM {DIM_CONTENT}
""").df()

features_df = features_df.merge(content_meta, on='content_hash_id', how='left')

print(f'=== Features built from warehouse ===')
print(f'Content items with enough history: {len(features_df):,}')
print()
print('Five features + label justification:')
print()
print('1. imp_prev30 — sum of impressions from Jan 30 to Feb 28')
print('   Knowable because: this window CLOSED on Feb 28, before March (the label window) started.')
print()
print('2. clk_prev30 — sum of clicks from the same pre-March window')
print('   Knowable because: same closed window as imp_prev30.')
print()
print('3. pos_prev30 — average GSC position from the pre-March window')
print('   Knowable because: same closed window. Position data is available as soon as the day ends.')
print()
print('4. days_with_imp_prev30 — how many days in the feature window had ≥1 impression')
print('   Knowable because: counts completed days only — Feb 28 is in the past by March 1.')
print()
print('5. content_age_days — computed from content_created_date in dim_content')
print('   Knowable because: set at publish time, never changes.')
print()
print('=== Label definition ===')
features_df['is_declining'] = (features_df['imp_last30'] < 0.8 * features_df['imp_prev30']).astype(int)
declining_rate = features_df['is_declining'].mean()
print(f'is_declining = 1 when imp_last30 < 0.8 × imp_prev30')
print(f'Declining rate in this slice: {declining_rate:.3f}')
print()
features_df.head(10)

=== Features built from warehouse ===
Content items with enough history: 11,161

Five features + label justification:

1. imp_prev30 — sum of impressions from Jan 30 to Feb 28
   Knowable because: this window CLOSED on Feb 28, before March (the label window) started.

2. clk_prev30 — sum of clicks from the same pre-March window
   Knowable because: same closed window as imp_prev30.

3. pos_prev30 — average GSC position from the pre-March window
   Knowable because: same closed window. Position data is available as soon as the day ends.

4. days_with_imp_prev30 — how many days in the feature window had ≥1 impression
   Knowable because: counts completed days only — Feb 28 is in the past by March 1.

5. content_age_days — computed from content_created_date in dim_content
   Knowable because: set at publish time, never changes.

=== Label definition ===
is_declining = 1 when imp_last30 < 0.8 × imp_prev30
Declining rate in this slice: 0.246



,content_hash_id,client_hash_id,imp_prev30,clk_prev30,pos_prev30,days_with_imp_prev30,imp_last30,content_age_days,word_count,is_declining
0,content_8a852c845b650ac2,client_e547b89c05043229,10604.0,84.0,5.982604,26,13305.0,226,2521,0
1,content_633e29a0bfb780f9,client_e547b89c05043229,1047.0,6.0,2.872968,9,1231.0,177,2647,0
2,content_6097e95a8b03dcf8,client_e547b89c05043229,1765.0,21.0,2.699550,18,1145.0,130,1215,1
3,content_a4c97a728f264492,client_e547b89c05043229,11862.0,87.0,4.133011,26,6867.0,130,2645,1
4,content_6fcf433caef4da4d,client_e547b89c05043229,33472.0,157.0,3.596657,29,27074.0,130,2710,0
5,content_3d5831138dc766c4,client_9958f0a7ae1df715,381.0,7.0,14.104781,19,478.0,338,2596,0
6,content_0f29eb1eece11b2f,client_9958f0a7ae1df715,1103.0,5.0,9.394374,21,1732.0,338,2434,0
7,content_ffadc4cfd8604d34,client_9958f0a7ae1df715,654.0,5.0,13.517709,29,1785.0,338,2621,0
8,content_5d8f15f56a60ffce,client_9958f0a7ae1df715,743.0,7.0,6.314942,9,610.0,417,2536,0
9,content_8b83c97749294243,client_23a62021009f63c4,4721.0,76.0,5.241347,25,4979.0,212,3388,0


In [5]:
# Verification D — The leakage trap
# Step 1: Train a model on clean features (no leakage)
# Step 2: Add a leaked column — impressions from the SAME window as the label
# Step 3: Watch the score jump
# Step 4: Delete the leaked column, keep honest baseline

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, accuracy_score

# Prepare clean data
clean_cols = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']
leak_cols = clean_cols + ['imp_last30']  # imp_last30 is the label window itself!

model_data = features_df.dropna(subset=leak_cols).copy()
X_clean = model_data[clean_cols]
X_leak = model_data[leak_cols]
y = model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X_clean, y, test_size=0.25, random_state=42, stratify=y)

# Clean model
clean_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clean_model.fit(X_tr, y_tr)
clean_pred = clean_model.predict(X_te)
clean_prec = precision_score(y_te, clean_pred)

print('=== STEP 1: Clean model (no leakage) ===')
print(f'Features used: {clean_cols}')
print(f'Precision (declining class): {clean_prec:.3f}')
print(f'Accuracy: {accuracy_score(y_te, clean_pred):.3f}')
print(f'Base rate: {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print()

# Now add the leaked column
X_tr_leak, X_te_leak, y_tr_leak, y_te_leak = train_test_split(
    X_leak, y, test_size=0.25, random_state=42, stratify=y
)

leak_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
leak_model.fit(X_tr_leak, y_tr_leak)
leak_pred = leak_model.predict(X_te_leak)
leak_prec = precision_score(y_te_leak, leak_pred)

print('=== STEP 2: Leaked model (imp_last30 added) ===')
print(f'Features used: {leak_cols}')
print(f'Precision (declining class): {leak_prec:.3f}')
print(f'Accuracy: {accuracy_score(y_te_leak, leak_pred):.3f}')
print()

print('=== STEP 3: Comparison ===')
print(f'Clean precision: {clean_prec:.3f}')
print(f'Leaked precision: {leak_prec:.3f}')
print(f'Jump: {leak_prec - clean_prec:.3f}')
print()
print('=== STEP 4: Delete leaked column ===')
print('Done — imp_last30 is removed from the feature set.')
print('The honest baseline is the clean model above.')
print()
print('Leakage verdict: If adding imp_last30 (which is half the label definition) ')
print('jumps precision, that column was leaking the answer. The model wasn\'t learning')
print('patterns — it was reading the grade before the exam.')

=== STEP 1: Clean model (no leakage) ===
Features used: ['imp_prev30', 'clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']
Precision (declining class): 0.536
Accuracy: 0.765
Base rate: 0.754

=== STEP 2: Leaked model (imp_last30 added) ===
Features used: ['imp_prev30', 'clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days', 'imp_last30']
Precision (declining class): 0.983
Accuracy: 0.971

=== STEP 3: Comparison ===
Clean precision: 0.536
Leaked precision: 0.983
Jump: 0.446

=== STEP 4: Delete leaked column ===
Done — imp_last30 is removed from the feature set.
The honest baseline is the clean model above.

Leakage verdict: If adding imp_last30 (which is half the label definition) 
jumps precision, that column was leaking the answer. The model wasn't learning
patterns — it was reading the grade before the exam.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Limitation 1: Unbalanced panel (history depth varies per client)

Not every client has the full 17 months of history. Some joined later. Some have GSC-only early history without GA4. If I define a global time window (e.g. "last 60 days for everyone"), clients with shorter history will drop out of the analysis silently. I must check `dim_clients.gsc_data_start` and `ga4_data_start` before setting any window.

### Limitation 2: GSC-only early rows (zero-filled GA4, not true zeros)

Rows before a client's GA4 data start have GA4 columns zero-filled with `ga4_data_available = FALSE`. A `SUM(sessions_90d) = 0` on those rows does not mean "no sessions" — it means "we don't have GA4 data for this period." Filtering with `ga4_data_available IS TRUE` is required, not optional.

### Limitation 3: The query table's window overlaps the label window

`fact_content_query_90d` covers a fixed 90-day window ending at the snapshot date. For a label defined in the last 30 days of that window, the query table is **not safe as a feature source** — its window contains the label period. Only the `prev_30` style columns from the daily fact are safe.

### Limitation 4: The label is a proxy, not an observed outcome

Same limitation as ML-03: `is_declining` compares two adjacent 30-day windows to measure "is this page declining right now?" It does not measure "will this page decline next month?" A true forward-looking label requires a different data pipeline that shifts the feature window backward and the label window forward.

### Limitation 5: NULL handling in ga4 availability

From the data dictionary: *"10 of 104 clients have NULL access flags in dim_clients."* Using `= FALSE` silently misses NULL rows — must use `IS NOT TRUE` instead.

In [6]:
# Section 4 — Demonstrate the unbalanced panel

# Show how history depth varies per client
client_history = con.sql(f"""
    SELECT 
        client_hash_id,
        access_profile,
        gsc_data_start,
        ga4_data_start,
        CASE 
            WHEN gsc_data_start IS NULL THEN 'no GSC'
            WHEN ga4_data_start IS NULL THEN 'GSC only'
            WHEN ga4_data_start > gsc_data_start THEN 'GSC + delayed GA4'
            ELSE 'GSC + GA4 from start'
        END AS profile_type
    FROM {DIM_CLIENTS}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('=== Client history depth (first 15) ===')
print(client_history.head(15).to_string(index=False))
print()
print('Profile type counts:')
print(client_history['profile_type'].value_counts().to_string())
print()
print(f"Clients with NULL gsc_data_start: {client_history['gsc_data_start'].isna().sum()}")
print(f"Clients with NULL ga4_data_start: {client_history['ga4_data_start'].isna().sum()}")
print()
print('Limitation: A global 60-day feature window will silently drop')
print('clients whose history started less than 60 days before the analysis period.')

=== Client history depth (first 15) ===
         client_hash_id access_profile gsc_data_start ga4_data_start      profile_type
client_9958f0a7ae1df715    gsc_and_ga4     2025-01-27     2025-10-29 GSC + delayed GA4
client_ff644d8251367cbb    gsc_and_ga4     2025-01-27     2025-10-29 GSC + delayed GA4
client_73cda7b4e4f265ea    gsc_and_ga4     2025-02-11     2026-03-24 GSC + delayed GA4
client_fef1a8f436438636    gsc_and_ga4     2025-03-11     2026-03-06 GSC + delayed GA4
client_62f4a7e64f5e0096       gsc_only     2025-06-07            NaT          GSC only
client_b10cb2997d0c7c86    gsc_and_ga4     2025-06-18     2025-11-15 GSC + delayed GA4
client_c182d11e4862a37d    gsc_and_ga4     2025-06-21     2026-02-20 GSC + delayed GA4
client_65de48885f4ef01b    gsc_and_ga4     2025-06-21     2026-02-19 GSC + delayed GA4
client_3197e6291363b4db    gsc_and_ga4     2025-06-29     2025-11-09 GSC + delayed GA4
client_625b6439094e23e4    gsc_and_ga4     2025-07-01     2026-02-19 GSC + delayed GA4
cli

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Leakage trap: added label-derived column → score jumped → deleted → honest baseline kept
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.